# ECG Image Digitization — EfficientNet-B0 Fast Training

**Goal:** Convert ECG paper images → 12-lead time-series signals with **>90% Pearson correlation**

**Architecture:** EfficientNet-B0 Encoder + Cross-Attention + Progressive 1D Decoder

### Why EfficientNet-B0 instead of DenseNet-121?
| Feature | DenseNet-121 | EfficientNet-B0 |
|---------|-------------|------------------|
| Backbone params | 7.0M | 5.3M |
| Output features | 1024 | 1280 |
| FLOPs (224×224) | 2.87B | 0.39B |
| CPU inference | Slow (dense concat) | **7× faster** (depthwise conv) |
| ImageNet Top-1 | 74.4% | **77.1%** |

EfficientNet-B0 uses **compound scaling** + **depthwise separable convolutions** making it dramatically faster on CPU while extracting richer features.

In [1]:
# Cell 1: Install dependencies (run once)
import subprocess, sys
pkgs = ['torch', 'torchvision', 'numpy', 'pandas', 'matplotlib', 'wfdb', 'opencv-python', 'Pillow', 'tqdm']
for pkg in pkgs:
    try:
        __import__(pkg.replace('-', '_').split('==')[0])
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
print('All packages ready.')

All packages ready.


In [2]:
# Cell 2: Imports
import os, time, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
import wfdb
from PIL import Image
from pathlib import Path
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torchvision.models as models
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader, random_split

warnings.filterwarnings('ignore')

# Device setup
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
print(f'PyTorch: {torch.__version__}')

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

Device: cpu
PyTorch: 2.10.0+cpu


## Step 1: Configuration
All hyperparameters in one place for easy tuning.

In [13]:

# Cell 3: Configuration
CFG = {
    # Paths
    'ptbxl_dir': r'ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3'
                 r'\ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3',
    'image_dir': 'ecg_data',
    'output_dir': 'outputs',
    'checkpoint_dir': 'digitization_checkpoints_effnet',
    
    # Data
    'num_samples': 4000,        # Number of samples to load (from PTB-XL)
    'sampling_rate': 100,       # Hz
    'signal_length': 1000,      # 10 sec × 100 Hz
    'num_leads': 12,
    'img_size': 224,
    'train_split': 0.85,
    
    # Training  ← REDUCED for speed while targeting >90%
    'epochs': 10,               # ↓ from 20 (aggressive schedule compensates)
    'batch_size': 32,
    'lr_encoder': 1e-4,         # ↑ from 5e-5  — faster encoder adaptation
    'lr_decoder': 2e-3,         # ↑ from 1.5e-3 — faster decoder convergence
    'weight_decay': 1e-4,
    'grad_clip': 1.0,
    'warmup_pct': 0.10,         # ↓ from 0.15 — shorter warmup, more training time
    'freeze_encoder_epochs': 1, # ↓ from 2 — unfreeze encoder sooner

    # Model
    'dropout': 0.12,
    'decoder_channels': [64, 48, 32, 16],
    
    # Loss weights (curriculum) — more aggressive
    'mse_weight': 1.0,
    'corr_weight_max': 2.0,     # ↑ from 1.5 — stronger correlation signal
    'shape_weight_max': 0.5,    # ↑ from 0.4

    # Early stopping
    'patience': 5,              # ↓ from 8 — stop sooner if stuck
    'target_corr': 0.90,
}

os.makedirs(CFG['output_dir'], exist_ok=True)
os.makedirs(CFG['checkpoint_dir'], exist_ok=True)

print('Configuration:')
for k, v in CFG.items():
    if not k.endswith('_dir'):
        print(f'  {k}: {v}')


Configuration:
  num_samples: 4000
  sampling_rate: 100
  signal_length: 1000
  num_leads: 12
  img_size: 224
  train_split: 0.85
  epochs: 10
  batch_size: 32
  lr_encoder: 0.0001
  lr_decoder: 0.002
  weight_decay: 0.0001
  grad_clip: 1.0
  warmup_pct: 0.1
  freeze_encoder_epochs: 1
  dropout: 0.12
  decoder_channels: [64, 48, 32, 16]
  mse_weight: 1.0
  corr_weight_max: 2.0
  shape_weight_max: 0.5
  patience: 5
  target_corr: 0.9


## Step 2: EfficientNet-B0 Digitization Model

**Architecture overview:**
```
ECG Image (224×224×3)
    │
    ▼
EfficientNet-B0 Encoder (pretrained) → (B, 1280, 7, 7)
    │
    ▼
Channel Projection → (B, 512, 7, 7) → flatten → (B, 49, 512)
    │
    ▼
Cross-Attention: 12 lead queries attend to 49 spatial features → (B, 12, 512)
    │
    ▼
Self-Attention: inter-lead dependencies → (B, 12, 512)
    │
    ▼
Progressive 1D Decoder: ConvTranspose1d upsampling
  (B×12, 64, 16) → (B×12, 1, 1024) → trim → (B, 12, 1000)
    │
    ▼
Cross-lead Refinement → (B, 12, 1000)
```

In [14]:
# Cell 4: EfficientNet-B0 ECG Digitization Model

class EfficientNetECGDigitization(nn.Module):
    """
    ECG Image → 12-Lead Signal Digitization
    Encoder:  EfficientNet-B0 (pretrained, 5.3M params, 7× faster than DenseNet-121)
    Decoder:  Cross-Attention + Progressive ConvTranspose1d
    """

    def __init__(self, signal_length=1000, num_leads=12, dropout=0.12):
        super().__init__()
        self.signal_length = signal_length
        self.num_leads = num_leads

        # ── Encoder: EfficientNet-B0 ──
        effnet = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
        self.encoder = effnet.features          # → (B, 1280, 7, 7)

        # Channel reduction: 1280 → 512 (saves memory + compute)
        self.channel_proj = nn.Sequential(
            nn.BatchNorm2d(1280),
            nn.SiLU(inplace=True),
            nn.Conv2d(1280, 512, kernel_size=1, bias=False),
            nn.BatchNorm2d(512),
            nn.SiLU(inplace=True),
        )

        # Positional encoding for 7×7 = 49 spatial positions
        self.spatial_pos = nn.Parameter(torch.randn(1, 49, 512) * 0.02)

        # ── Cross-Attention: lead queries attend to spatial features ──
        self.lead_queries = nn.Parameter(torch.randn(1, num_leads, 512) * 0.02)

        self.cross_attn = nn.MultiheadAttention(
            embed_dim=512, num_heads=8, dropout=dropout, batch_first=True
        )
        self.cross_norm = nn.LayerNorm(512)

        # Self-attention: inter-lead relationships (I, II → III = I - II, etc.)
        self.self_attn = nn.MultiheadAttention(
            embed_dim=512, num_heads=8, dropout=dropout, batch_first=True
        )
        self.self_norm = nn.LayerNorm(512)

        # Feed-forward network
        self.ffn = nn.Sequential(
            nn.Linear(512, 1024),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(1024, 512),
            nn.Dropout(dropout),
        )
        self.ffn_norm = nn.LayerNorm(512)

        # ── Progressive 1D Decoder ──
        # Per-lead: 512 → reshape to (64, 16) → upsample to 1024 → trim to 1000
        self.seq_proj = nn.Sequential(
            nn.Linear(512, 64 * 16),
            nn.GELU(),
        )

        # Progressive upsampling: 16 → 64 → 256 → 1024
        self.decoder = nn.Sequential(
            nn.ConvTranspose1d(64, 48, kernel_size=4, stride=4, padding=0),   # 16→64
            nn.BatchNorm1d(48),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),

            nn.ConvTranspose1d(48, 32, kernel_size=4, stride=4, padding=0),   # 64→256
            nn.BatchNorm1d(32),
            nn.GELU(),
            nn.Dropout(dropout * 0.5),

            nn.ConvTranspose1d(32, 16, kernel_size=4, stride=4, padding=0),   # 256→1024
            nn.BatchNorm1d(16),
            nn.GELU(),

            nn.Conv1d(16, 1, kernel_size=7, padding=3),                       # channels→1
        )

        # ── Cross-lead Refinement ──
        self.refinement = nn.Sequential(
            nn.Conv1d(num_leads, num_leads * 2, kernel_size=15, padding=7),
            nn.BatchNorm1d(num_leads * 2),
            nn.GELU(),
            nn.Conv1d(num_leads * 2, num_leads, kernel_size=7, padding=3),
            nn.Tanh(),
        )

        # Initialize decoder weights
        self._init_weights()

    def _init_weights(self):
        for m in [self.decoder, self.refinement]:
            for layer in m.modules():
                if isinstance(layer, (nn.ConvTranspose1d, nn.Conv1d)):
                    nn.init.kaiming_normal_(layer.weight, nonlinearity='relu')
                    if layer.bias is not None:
                        nn.init.zeros_(layer.bias)

    def forward(self, x):
        B = x.size(0)

        # 1) Encode image
        feat = self.encoder(x)                          # (B, 1280, 7, 7)
        feat = self.channel_proj(feat)                  # (B, 512, 7, 7)

        # 2) Flatten spatial → sequence + positional encoding
        spatial = feat.flatten(2).permute(0, 2, 1)      # (B, 49, 512)
        spatial = spatial + self.spatial_pos

        # 3) Cross-attention: lead queries attend to spatial features
        queries = self.lead_queries.expand(B, -1, -1)   # (B, 12, 512)
        attn_out, _ = self.cross_attn(queries, spatial, spatial)
        leads = self.cross_norm(queries + attn_out)     # (B, 12, 512)

        # 4) Self-attention: inter-lead dependencies
        self_out, _ = self.self_attn(leads, leads, leads)
        leads = self.self_norm(leads + self_out)        # (B, 12, 512)

        # 5) FFN
        ffn_out = self.ffn(leads)
        leads = self.ffn_norm(leads + ffn_out)          # (B, 12, 512)

        # 6) Progressive decode (shared decoder for all leads)
        seq = self.seq_proj(leads)                      # (B, 12, 64*16)
        seq = seq.view(B * self.num_leads, 64, 16)      # (B*12, 64, 16)
        decoded = self.decoder(seq)                     # (B*12, 1, 1024)
        decoded = decoded[:, 0, :self.signal_length]    # (B*12, 1000)
        signals = decoded.view(B, self.num_leads, -1)   # (B, 12, 1000)

        # 7) Cross-lead refinement
        signals = self.refinement(signals)              # (B, 12, 1000)
        return signals


# Build & verify model
print('Building EfficientNet-B0 ECG Digitization Model...')
model = EfficientNetECGDigitization(
    signal_length=CFG['signal_length'],
    num_leads=CFG['num_leads'],
    dropout=CFG['dropout']
).to(device)

# Count parameters
enc_params = sum(p.numel() for p in model.encoder.parameters())
dec_params = sum(p.numel() for p in model.parameters()) - enc_params
total_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'  Encoder params:   {enc_params:>10,}')
print(f'  Decoder params:   {dec_params:>10,}')
print(f'  Total params:     {total_params:>10,}')
print(f'  Trainable params: {trainable:>10,}')

# Verify forward pass
with torch.no_grad():
    dummy = torch.randn(2, 3, 224, 224, device=device)
    out = model(dummy)
    print(f'  Input:  {dummy.shape}')
    print(f'  Output: {out.shape}')
    assert out.shape == (2, 12, 1000), 'Shape mismatch!'

print('Model built successfully!')

Building EfficientNet-B0 ECG Digitization Model...
  Encoder params:    4,007,548
  Decoder params:    4,397,221
  Total params:      8,404,769
  Trainable params:  8,404,769
  Input:  torch.Size([2, 3, 224, 224])
  Output: torch.Size([2, 12, 1000])
Model built successfully!


## Step 3: Loss Function with Curriculum Learning

**Three loss components:**
1. **MSE Loss** — point-wise reconstruction (always active)
2. **Correlation Loss** — `1 - Pearson r` penalizes shape mismatch (activated epoch 3+)
3. **Shape Loss** — derivative matching for waveform morphology (activated epoch 7+)

In [15]:

# Cell 5: Loss Function

class CurriculumDigitizationLoss(nn.Module):
    """
    Combined loss with curriculum scheduling.
    Aggressive schedule for 10-epoch training:
      epoch 1:   MSE only
      epoch 2+:  MSE + Correlation (ramp)
      epoch 4+:  MSE + Correlation + Shape (ramp)
    """

    def __init__(self, mse_w=1.0, corr_w_max=2.0, shape_w_max=0.5):
        super().__init__()
        self.mse_w = mse_w
        self.corr_w_max = corr_w_max
        self.shape_w_max = shape_w_max

    def pearson_corr(self, pred, target):
        """Vectorized Pearson correlation across all leads."""
        pred_c = pred - pred.mean(dim=2, keepdim=True)
        tgt_c = target - target.mean(dim=2, keepdim=True)
        num = (pred_c * tgt_c).sum(dim=2)
        den = torch.sqrt((pred_c ** 2).sum(dim=2) * (tgt_c ** 2).sum(dim=2) + 1e-8)
        corr = num / den
        return corr.mean()

    def forward(self, pred, target, epoch=1):
        # ── MSE (always active) ──
        mse = F.mse_loss(pred, target)

        # ── Correlation loss (ramp in from epoch 2 — earlier than before) ──
        corr = self.pearson_corr(pred, target)
        if epoch >= 2:
            ramp = min((epoch - 1) / 3, 1.0)       # full ramp by epoch 4
            corr_w = self.corr_w_max * ramp
        else:
            corr_w = 0.0
        corr_loss = (1.0 - corr) * corr_w

        # ── Shape (derivative) loss (ramp in from epoch 4 — earlier than before) ──
        pred_diff = pred[:, :, 1:] - pred[:, :, :-1]
        tgt_diff  = target[:, :, 1:] - target[:, :, :-1]
        shape_mse = F.mse_loss(pred_diff, tgt_diff)
        if epoch >= 4:
            ramp_s = min((epoch - 3) / 3, 1.0)     # full ramp by epoch 6
            shape_w = self.shape_w_max * ramp_s
        else:
            shape_w = 0.0
        shape_loss = shape_mse * shape_w

        total = self.mse_w * mse + corr_loss + shape_loss

        return total, {
            'total': total.item(),
            'mse': mse.item(),
            'corr': corr.item(),
            'shape': shape_mse.item(),
            'corr_w': corr_w,
            'shape_w': shape_w,
        }

criterion = CurriculumDigitizationLoss(
    mse_w=CFG['mse_weight'],
    corr_w_max=CFG['corr_weight_max'],
    shape_w_max=CFG['shape_weight_max']
)
print('Loss function ready — aggressive curriculum (corr@epoch2, shape@epoch4)')


Loss function ready — aggressive curriculum (corr@epoch2, shape@epoch4)


## Step 4: Dataset — Dynamic Image Generation from PTB-XL

Loads raw PTB-XL signals, renders them as ECG images on-the-fly, and pairs them with the ground-truth signals. This avoids needing pre-generated image files.

In [6]:
# Cell 6: Dataset

class FastECGDigitizationDataset(Dataset):
    """
    Generates (ECG_image, signal) pairs on-the-fly from PTB-XL.
    Images are rendered dynamically with randomized styles for augmentation.
    """

    def __init__(self, ptbxl_dir, num_samples=3000, sampling_rate=100,
                 signal_length=1000, transform=None, augment=False):
        self.ptbxl_dir = ptbxl_dir
        self.sampling_rate = sampling_rate
        self.signal_length = signal_length
        self.transform = transform
        self.augment = augment

        db = pd.read_csv(os.path.join(ptbxl_dir, 'ptbxl_database.csv'))
        self.database = db.head(num_samples).reset_index(drop=True)
        print(f'  Dataset: {len(self.database)} samples @ {sampling_rate} Hz')

    def __len__(self):
        return len(self.database)

    def __getitem__(self, idx):
        row = self.database.iloc[idx]
        fname = row['filename_lr'] if self.sampling_rate == 100 else row['filename_hr']
        sig_path = os.path.join(self.ptbxl_dir, fname)

        try:
            record = wfdb.rdrecord(sig_path)
            signal = record.p_signal.T      # (12, samples)
            signal = self._normalize(signal)
            pil_img = self._render_ecg(signal)

            if self.transform:
                image = self.transform(pil_img)
            else:
                image = torch.from_numpy(np.array(pil_img)).permute(2, 0, 1).float() / 255.0

            sig_tensor = torch.from_numpy(signal[:, :self.signal_length].copy()).float()
            return image, sig_tensor

        except Exception as e:
            return torch.zeros(3, 224, 224), torch.zeros(12, self.signal_length)

    def _normalize(self, signal):
        """Per-lead robust normalization to [-1, 1]."""
        out = np.zeros_like(signal)
        for i in range(signal.shape[0]):
            s = signal[i] - np.mean(signal[i])
            lo, hi = np.percentile(s, [1, 99])
            if hi - lo > 1e-6:
                s = 2 * (s - lo) / (hi - lo) - 1
                s = np.clip(s, -1, 1)
            out[i] = s
        return out

    def _render_ecg(self, signal, W=800, H=600):
        """Render 12-lead ECG as a realistic paper-style image."""
        img = np.ones((H, W, 3), dtype=np.uint8) * 255

        # Grid
        gc = (230, 210, 210) if (self.augment and np.random.rand() < 0.5) else (220, 220, 220)
        for x in range(0, W, 20):
            cv2.line(img, (x, 0), (x, H), gc, 1)
        for y in range(0, H, 20):
            cv2.line(img, (0, y), (W, y), gc, 1)

        n_leads = min(signal.shape[0], 12)
        lh = H // n_leads
        thickness = 2 if (self.augment and np.random.rand() < 0.3) else 1

        for li in range(n_leads):
            s = signal[li, :min(signal.shape[1], W)]
            y_off = li * lh + lh // 2
            y_scale = lh * 0.38
            pts = []
            for xi in range(len(s)):
                yi = int(np.clip(y_off - s[xi] * y_scale, 0, H - 1))
                pts.append((xi, yi))
            for j in range(len(pts) - 1):
                cv2.line(img, pts[j], pts[j + 1], (0, 0, 0), thickness)

        # Augmentation noise
        if self.augment:
            if np.random.rand() < 0.3:
                noise = np.random.randint(0, 8, img.shape, dtype=np.uint8)
                img = np.clip(img.astype(np.int16) + noise - 4, 0, 255).astype(np.uint8)
            if np.random.rand() < 0.2:
                img = cv2.GaussianBlur(img, (3, 3), 0)

        pil = Image.fromarray(img).resize((224, 224), Image.LANCZOS)
        return pil


print('Dataset class ready.')

Dataset class ready.


## Step 5: Prepare DataLoaders

In [7]:
# Cell 7: Create dataloaders

# Training transforms with augmentation
train_transform = T.Compose([
    T.RandomApply([T.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.1)], p=0.4),
    T.RandomApply([T.GaussianBlur(kernel_size=3)], p=0.15),
    T.RandomAffine(degrees=1.5, translate=(0.01, 0.01), scale=(0.97, 1.03)),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

val_transform = T.Compose([
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

print('Loading PTB-XL dataset...')
full_dataset = FastECGDigitizationDataset(
    ptbxl_dir=CFG['ptbxl_dir'],
    num_samples=CFG['num_samples'],
    sampling_rate=CFG['sampling_rate'],
    signal_length=CFG['signal_length'],
    transform=train_transform,
    augment=True
)

# Split
n_train = int(CFG['train_split'] * len(full_dataset))
n_val = len(full_dataset) - n_train
train_ds, val_ds = random_split(full_dataset, [n_train, n_val],
                                 generator=torch.Generator().manual_seed(42))

# Override val transform (no augmentation)
class ValWrapper(Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform
    def __len__(self):
        return len(self.subset)
    def __getitem__(self, idx):
        # Get underlying dataset item, but we need to change transform
        orig_transform = self.subset.dataset.transform
        orig_augment = self.subset.dataset.augment
        self.subset.dataset.transform = self.transform
        self.subset.dataset.augment = False
        item = self.subset[idx]
        self.subset.dataset.transform = orig_transform
        self.subset.dataset.augment = orig_augment
        return item

val_ds_wrapped = ValWrapper(val_ds, val_transform)

train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True,
                          num_workers=0, pin_memory=False, drop_last=True)
val_loader = DataLoader(val_ds_wrapped, batch_size=CFG['batch_size'], shuffle=False,
                        num_workers=0, pin_memory=False)

print(f'  Train: {n_train} samples → {len(train_loader)} batches')
print(f'  Val:   {n_val} samples → {len(val_loader)} batches')
print('DataLoaders ready.')

Loading PTB-XL dataset...
  Dataset: 4000 samples @ 100 Hz
  Train: 3400 samples → 106 batches
  Val:   600 samples → 19 batches
DataLoaders ready.


## Step 6: Optimizer, Scheduler & Training Setup

- **Differential learning rates**: encoder (5e-5) vs decoder (1.5e-3)
- **OneCycleLR**: fast warm-up + cosine decay
- **Encoder freezing**: first 2 epochs freeze encoder, then unfreeze with low LR

In [16]:

# Cell 8: Optimizer & Scheduler
# FIX: Both param groups must be registered with OneCycleLR BEFORE it is created.
# Adding a new group mid-training causes KeyError: 'initial_lr'.
# Solution: encoder group starts in the optimizer with near-zero LR; requires_grad
# is False so no gradients are computed until we unfreeze it.

print(f'Freezing EfficientNet-B0 encoder for first {CFG["freeze_encoder_epochs"]} epoch(s)...')
for p in model.encoder.parameters():
    p.requires_grad = False

decoder_params = [p for n, p in model.named_parameters() if not n.startswith('encoder')]
encoder_params = list(model.encoder.parameters())

# Register BOTH groups upfront so OneCycleLR tracks them from the start.
# Encoder starts at near-zero LR; it will ramp up once unfrozen.
optimizer = optim.AdamW([
    {'params': decoder_params, 'lr': CFG['lr_decoder']},
    {'params': encoder_params,  'lr': CFG['lr_encoder']},
], betas=(0.9, 0.999), weight_decay=CFG['weight_decay'])

# OneCycleLR — pass max_lr as a list matching the two param groups
steps_per_epoch = len(train_loader)
total_steps = CFG['epochs'] * steps_per_epoch

scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer,
    max_lr=[CFG['lr_decoder'], CFG['lr_encoder']],   # one per param group
    total_steps=total_steps,
    pct_start=CFG['warmup_pct'],
    anneal_strategy='cos',
    div_factor=10,
    final_div_factor=100,
)

trainable_now = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable parameters now (encoder frozen): {trainable_now:,}')
print(f'Steps/epoch: {steps_per_epoch},  Total steps: {total_steps}')
print('Optimizer & scheduler ready (KeyError fixed — both groups pre-registered).')


Freezing EfficientNet-B0 encoder for first 1 epoch(s)...
Trainable parameters now (encoder frozen): 4,397,221
Steps/epoch: 106,  Total steps: 1060
Optimizer & scheduler ready (KeyError fixed — both groups pre-registered).


## Step 7: Training Loop

**Key features:**
- Curriculum loss scheduling (MSE → +Correlation → +Shape)
- Encoder unfreezing at epoch 3 with differential LR
- Gradient clipping for stability
- Early stopping with patience
- Per-epoch Pearson correlation tracking

In [ ]:

# Cell 9: Training Loop

def pearson_correlation_batch(pred, target):
    """Compute mean Pearson correlation across batch and leads."""
    pred_c = pred - pred.mean(dim=2, keepdim=True)
    tgt_c = target - target.mean(dim=2, keepdim=True)
    num = (pred_c * tgt_c).sum(dim=2)
    den = torch.sqrt((pred_c ** 2).sum(dim=2) * (tgt_c ** 2).sum(dim=2) + 1e-8)
    return (num / den).mean().item()


# History
history = {
    'train_loss': [], 'val_loss': [],
    'train_mse': [], 'val_mse': [],
    'train_mae': [], 'val_mae': [],
    'train_corr': [], 'val_corr': [],
    'lr': [],
}

best_val_loss = float('inf')
best_val_corr = 0.0
patience_counter = 0
encoder_unfrozen = False

print('=' * 80)
print('STARTING TRAINING — EfficientNet-B0 + Progressive Decoder')
print(f'  Epochs: {CFG["epochs"]}  |  Batch: {CFG["batch_size"]}  |  Target: >{CFG["target_corr"]:.0%} correlation')
print('=' * 80)

start_time = time.time()

for epoch in range(1, CFG['epochs'] + 1):
    epoch_start = time.time()

    # ── Unfreeze encoder (just re-enable requires_grad; optimizer already has the group) ──
    if epoch == CFG['freeze_encoder_epochs'] + 1 and not encoder_unfrozen:
        print(f'\n>>> Unfreezing EfficientNet-B0 encoder at epoch {epoch}.')
        for p in model.encoder.parameters():
            p.requires_grad = True
        encoder_unfrozen = True
        trainable_now = sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(f'    Trainable parameters: {trainable_now:,}')

    # ──── TRAIN ────
    model.train()
    t_loss, t_mse, t_mae, t_corr = 0, 0, 0, 0
    n_batches = 0

    pbar = tqdm(train_loader, desc=f'Epoch {epoch:02d}/{CFG["epochs"]:02d} [Train]',
                leave=False, ncols=100)
    for images, signals in pbar:
        images, signals = images.to(device), signals.to(device)

        optimizer.zero_grad()
        pred = model(images)
        loss, ld = criterion(pred, signals, epoch=epoch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), CFG['grad_clip'])
        optimizer.step()
        scheduler.step()

        with torch.no_grad():
            mse = F.mse_loss(pred, signals).item()
            mae = F.l1_loss(pred, signals).item()
            corr = pearson_correlation_batch(pred, signals)

        t_loss += ld['total']; t_mse += mse; t_mae += mae; t_corr += corr
        n_batches += 1
        pbar.set_postfix(loss=f'{ld["total"]:.4f}', corr=f'{corr:.3f}')

    t_loss /= n_batches; t_mse /= n_batches; t_mae /= n_batches; t_corr /= n_batches

    # ──── VALIDATE ────
    model.eval()
    v_loss, v_mse, v_mae, v_corr = 0, 0, 0, 0
    v_batches = 0

    with torch.no_grad():
        for images, signals in val_loader:
            images, signals = images.to(device), signals.to(device)
            pred = model(images)
            loss, ld = criterion(pred, signals, epoch=epoch)
            mse = F.mse_loss(pred, signals).item()
            mae = F.l1_loss(pred, signals).item()
            corr = pearson_correlation_batch(pred, signals)
            v_loss += ld['total']; v_mse += mse; v_mae += mae; v_corr += corr
            v_batches += 1

    v_loss /= v_batches; v_mse /= v_batches; v_mae /= v_batches; v_corr /= v_batches

    # ── Record history ──
    current_lr = optimizer.param_groups[0]['lr']
    history['train_loss'].append(t_loss); history['val_loss'].append(v_loss)
    history['train_mse'].append(t_mse); history['val_mse'].append(v_mse)
    history['train_mae'].append(t_mae); history['val_mae'].append(v_mae)
    history['train_corr'].append(t_corr); history['val_corr'].append(v_corr)
    history['lr'].append(current_lr)

    elapsed = time.time() - epoch_start

    # ── Print summary ──
    if epoch < 2:
        phase_tag = '[MSE only]'
    elif epoch < 4:
        phase_tag = f'[MSE+Corr w={ld["corr_w"]:.2f}]'
    else:
        phase_tag = f'[MSE+Corr+Shape w={ld["corr_w"]:.2f}/{ld["shape_w"]:.2f}]'

    print(f'Epoch {epoch:02d} {phase_tag} ({elapsed:.0f}s)  '
          f'Train: loss={t_loss:.4f} corr={t_corr:.4f}  |  '
          f'Val: loss={v_loss:.4f} corr={v_corr:.4f} mse={v_mse:.4f}')

    # ── Save best model ──
    improved = False
    if v_corr > best_val_corr:
        best_val_corr = v_corr
        improved = True
    if v_loss < best_val_loss:
        best_val_loss = v_loss
        improved = True

    if improved:
        patience_counter = 0
        ckpt_path = os.path.join(CFG['checkpoint_dir'], 'best_effnet_digitization.pth')
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_loss': v_loss,
            'val_corr': v_corr,
            'history': history,
            'config': CFG,
        }, ckpt_path)
        print(f'    >>> Saved best model (corr={v_corr:.4f}, loss={v_loss:.4f})')
    else:
        patience_counter += 1

    if v_corr >= CFG['target_corr']:
        print(f'    TARGET REACHED! Pearson correlation = {v_corr:.4f} >= {CFG["target_corr"]}')

    # ── Save periodic checkpoint ──
    if epoch % 5 == 0:
        path = os.path.join(CFG['checkpoint_dir'], f'checkpoint_epoch_{epoch}.pth')
        torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                    'val_corr': v_corr, 'history': history}, path)

    # ── Early stopping ──
    if patience_counter >= CFG['patience']:
        print(f'\nEarly stopping at epoch {epoch} (no improvement for {CFG["patience"]} epochs)')
        break

total_time = time.time() - start_time
print('\n' + '=' * 80)
print(f'TRAINING COMPLETE in {total_time/60:.1f} minutes')
print(f'  Best val correlation: {best_val_corr:.4f}')
print(f'  Best val loss:        {best_val_loss:.4f}')
print(f'  Target (>90%):        {"ACHIEVED" if best_val_corr >= 0.90 else "NOT YET — try more epochs"}')
print('=' * 80)


STARTING TRAINING — EfficientNet-B0 + Progressive Decoder
  Epochs: 10  |  Batch: 32  |  Target: >90% correlation


Epoch 01/10 [Train]:   0%|                                                  | 0/106 [00:00<?, ?it/s]

## Step 8: Training Curves Visualization

In [ ]:
# Cell 10: Plot training curves

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
epochs_range = range(1, len(history['train_loss']) + 1)

# Loss
axes[0, 0].plot(epochs_range, history['train_loss'], 'b-o', label='Train', markersize=4)
axes[0, 0].plot(epochs_range, history['val_loss'], 'r-o', label='Val', markersize=4)
axes[0, 0].set_title('Loss', fontweight='bold', fontsize=13)
axes[0, 0].set_xlabel('Epoch'); axes[0, 0].legend(); axes[0, 0].grid(alpha=0.3)

# Correlation (ACCURACY)
axes[0, 1].plot(epochs_range, history['train_corr'], 'b-o', label='Train', markersize=4)
axes[0, 1].plot(epochs_range, history['val_corr'], 'r-o', label='Val', markersize=4)
axes[0, 1].axhline(y=0.90, color='green', linestyle='--', alpha=0.7, label='Target 90%')
axes[0, 1].set_title('Pearson Correlation (Accuracy)', fontweight='bold', fontsize=13)
axes[0, 1].set_xlabel('Epoch'); axes[0, 1].legend(); axes[0, 1].grid(alpha=0.3)
axes[0, 1].set_ylim(0, 1.05)

# MSE
axes[1, 0].plot(epochs_range, history['train_mse'], 'b-o', label='Train', markersize=4)
axes[1, 0].plot(epochs_range, history['val_mse'], 'r-o', label='Val', markersize=4)
axes[1, 0].set_title('MSE', fontweight='bold', fontsize=13)
axes[1, 0].set_xlabel('Epoch'); axes[1, 0].legend(); axes[1, 0].grid(alpha=0.3)

# Learning Rate
axes[1, 1].plot(epochs_range, history['lr'], 'g-o', markersize=4)
axes[1, 1].set_title('Learning Rate', fontweight='bold', fontsize=13)
axes[1, 1].set_xlabel('Epoch'); axes[1, 1].grid(alpha=0.3)
axes[1, 1].set_yscale('log')

plt.suptitle('EfficientNet-B0 ECG Digitization Training', fontweight='bold', fontsize=15, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(CFG['output_dir'], 'effnet_training_curves.png'), dpi=200, bbox_inches='tight')
plt.show()
print('Training curves saved.')

## Step 9: Test Inference — Visualize Predictions vs Ground Truth

In [ ]:
# Cell 11: Inference on validation samples

# Load best model
ckpt_path = os.path.join(CFG['checkpoint_dir'], 'best_effnet_digitization.pth')
if os.path.exists(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    print(f'Loaded best model from epoch {ckpt["epoch"]} (corr={ckpt["val_corr"]:.4f})')

model.eval()

# Grab a batch from validation
val_iter = iter(val_loader)
images, signals = next(val_iter)
images, signals = images.to(device), signals.to(device)

with torch.no_grad():
    pred = model(images)

# Calculate per-sample correlation
pred_np = pred.cpu().numpy()
sig_np = signals.cpu().numpy()

lead_names = ['I', 'II', 'III', 'aVR', 'aVL', 'aVF', 'V1', 'V2', 'V3', 'V4', 'V5', 'V6']

# Plot first 3 samples, 4 leads each
n_show = min(3, pred_np.shape[0])
show_leads = [0, 1, 6, 7]  # I, II, V1, V2

fig, axes = plt.subplots(n_show, len(show_leads), figsize=(20, 3.5 * n_show))
if n_show == 1:
    axes = axes[np.newaxis, :]

for si in range(n_show):
    for li_idx, li in enumerate(show_leads):
        ax = axes[si, li_idx]
        ax.plot(sig_np[si, li], 'b-', alpha=0.7, label='Ground Truth', linewidth=1.2)
        ax.plot(pred_np[si, li], 'r-', alpha=0.7, label='Predicted', linewidth=1.2)

        # Per-lead correlation
        r = np.corrcoef(sig_np[si, li], pred_np[si, li])[0, 1]
        ax.set_title(f'Sample {si+1} — Lead {lead_names[li]}  (r={r:.3f})',
                     fontsize=10, fontweight='bold')
        ax.legend(fontsize=7, loc='upper right')
        ax.grid(alpha=0.2)
        ax.set_xlabel('Sample')

plt.suptitle('EfficientNet-B0 Digitization: Ground Truth vs Predicted',
             fontweight='bold', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(CFG['output_dir'], 'effnet_inference_comparison.png'), dpi=200, bbox_inches='tight')
plt.show()

# Overall metrics
sample_corrs = []
for si in range(pred_np.shape[0]):
    for li in range(12):
        r = np.corrcoef(sig_np[si, li], pred_np[si, li])[0, 1]
        if not np.isnan(r):
            sample_corrs.append(r)

print(f'\nInference results on {pred_np.shape[0]} val samples:')
print(f'  Mean Pearson correlation: {np.mean(sample_corrs):.4f}')
print(f'  Median correlation:       {np.median(sample_corrs):.4f}')
print(f'  Min  correlation:         {np.min(sample_corrs):.4f}')
print(f'  Max  correlation:         {np.max(sample_corrs):.4f}')
print(f'  Leads with r > 0.90:      {sum(1 for r in sample_corrs if r > 0.90)}/{len(sample_corrs)}')

## Step 10: Save Final Model & Training Report

In [ ]:
# Cell 12: Save results

# Save training history
history_df = pd.DataFrame(history)
history_df.index = range(1, len(history_df) + 1)
history_df.index.name = 'epoch'
history_path = os.path.join(CFG['output_dir'], 'effnet_training_history.csv')
history_df.to_csv(history_path)
print(f'Training history saved to {history_path}')

# Save final model for inference pipeline
final_model_path = os.path.join(CFG['output_dir'], 'ecg_digitization_effnet_final.pth')
torch.save({
    'model_state_dict': model.state_dict(),
    'config': CFG,
    'best_val_corr': best_val_corr,
    'best_val_loss': best_val_loss,
    'architecture': 'EfficientNet-B0 + CrossAttention + Progressive1DDecoder',
}, final_model_path)
print(f'Final model saved to {final_model_path}')

# Print summary report
print('\n' + '=' * 80)
print('TRAINING SUMMARY REPORT')
print('=' * 80)
print(f'  Architecture:      EfficientNet-B0 + CrossAttention + Progressive Decoder')
print(f'  Total epochs:      {len(history["train_loss"])}')
print(f'  Dataset size:      {CFG["num_samples"]} samples ({n_train} train / {n_val} val)')
print(f'  Batch size:        {CFG["batch_size"]}')
print(f'  Best val loss:     {best_val_loss:.4f}')
print(f'  Best val corr:     {best_val_corr:.4f}  ({best_val_corr*100:.1f}%)')
print(f'  Target (90%):      {"ACHIEVED" if best_val_corr >= 0.90 else "IN PROGRESS"}')
print('=' * 80)

# Display history table
print('\nPer-epoch summary:')
display_cols = ['train_loss', 'val_loss', 'train_corr', 'val_corr', 'lr']
print(history_df[display_cols].to_string(float_format='{:.4f}'.format))

## Step 11: Tips to Improve Further

If the correlation is close but not yet >90%, try:

1. **Increase `num_samples`** from 4000 → 6000+ (more data = better generalization)
2. **Increase `epochs`** to 30-40 (the model may need more time to converge on CPU)
3. **Lower `lr_decoder`** to 8e-4 for finer optimization in later epochs
4. **Increase `corr_weight_max`** to 2.0 to put more emphasis on correlation
5. **Reduce `batch_size`** to 16 for more gradient updates per epoch

To apply these quickly, change the `CFG` dictionary in Cell 3 and re-run from Cell 7 onwards.